# GPU Check — Phase 00, Lesson 03

Verifies GPU access and benchmarks CPU vs GPU matrix multiplication.

**Before running:** Enable the GPU runtime in Colab.
> Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save

In [ ]:
# Install PyTorch if not already available (Colab usually has it pre-installed)
try:
    import torch
    print(f"PyTorch {torch.__version__} already available")
except ImportError:
    %pip install torch --quiet
    import torch
    print(f"Installed PyTorch {torch.__version__}")

## 1. GPU Info

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("\nNo GPU detected.")
    print("Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"Memory          : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute cap.    : {props.major}.{props.minor}")

## 2. CPU vs GPU Benchmark

Multiplies two 4000×4000 matrices on CPU, then on GPU, and compares the time.

In [ ]:
import time

if not torch.cuda.is_available():
    print("Skip: no GPU available. Enable GPU runtime and re-run.")
else:
    size = 4000
    a = torch.randn(size, size)
    b = torch.randn(size, size)

    # CPU
    start = time.time()
    _ = a @ b
    cpu_time = time.time() - start
    print(f"CPU matrix multiply ({size}x{size}): {cpu_time:.3f}s")

    # GPU
    a_gpu = a.to("cuda")
    b_gpu = b.to("cuda")
    torch.cuda.synchronize()  # wait for transfer to complete

    start = time.time()
    _ = a_gpu @ b_gpu
    torch.cuda.synchronize()  # wait for kernel to finish
    gpu_time = time.time() - start
    print(f"GPU matrix multiply ({size}x{size}): {gpu_time:.3f}s")
    print(f"Speedup: {cpu_time / gpu_time:.0f}x")

## 3. Estimated Max Model Size (fp16)

In [ ]:
if not torch.cuda.is_available():
    print("Skip: no GPU available.")
else:
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    params_billions = (vram_gb * 1e9 / 2) / 1e9  # 2 bytes per fp16 param
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"Max model size  : ~{params_billions:.0f}B parameters (fp16, weights only)")
    print()
    print("Note: inference needs ~2x weights (KV cache + activations),")
    print("      fine-tuning needs ~4-6x weights (optimizer states + gradients).")